# Model testing

Load data and convert

In [1]:
import pandas as pd
import numpy as np
import os
os.environ["OMP_NUM_THREADS"] = "7"


train_df = pd.read_parquet("data/train_data.parquet")
test_df = pd.read_parquet("data/test_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")
test_df.index = pd.to_numeric(test_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")
test_df = test_df.drop(columns=cols_to_drop, errors="ignore")


In [2]:
train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"].astype(int)
train_full_y_reg = train_df["target_annual_roi"]

test_full_X = test_df.drop(["target", "target_annual_roi"], axis=1)
test_full_y_cat = test_df["target"].astype(int)
test_full_y_reg = test_df["target_annual_roi"]

cols_to_drop = ["issue_d", "earliest_cr_line"]

train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")
test_full_X = test_full_X.drop(columns=cols_to_drop, errors="ignore")

In [3]:
train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

## Classification

In [4]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="binary", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor_1k_cat = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="mean")
).set_output(transform="pandas")

imputed_numeric_preprocessor_10k_cat = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

imputed_numeric_preprocessor_100k_cat = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="mean")
).set_output(transform="pandas")

passthrough_preprocessor = "passthrough"

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


In [5]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from ngboost import NGBClassifier
from tabpfn_client import TabPFNClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeRegressor
from ngboost.distns import Bernoulli


models_no_tuning_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(random_state=42, enable_categorical=True)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(random_state=42)),
    "CatBoost": (passthrough_preprocessor, CatBoostClassifier(random_state=42, verbose=0, cat_features=cols_to_nominal_cat)),
    "NGBoost": (numeric_preprocessor, NGBClassifier(random_state=42, verbose=False)),
    "GBM": (imputed_numeric_preprocessor_1k_cat, GradientBoostingClassifier(random_state=42)),
    "HistGBM": (numeric_preprocessor, HistGradientBoostingClassifier(random_state=42)),
    "Dummy - Most Frequent": (numeric_preprocessor, DummyClassifier(strategy="prior")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNClassifier())
}

In [6]:
xgboost_1k_cat_params = {'booster': 'gbtree', 'learning_rate': 0.049961658713390054, 'min_split_loss': 3.1792890444034416, 'max_depth': 4, 'min_child_weight': 0.03446885315674068, 'max_delta_step': 8.818441265852027, 'colsample_bytree': 0.6612602163953242, 'colsample_bylevel': 0.4402275230048245, 'colsample_bynode': 0.44195112565215633, 'reg_lambda': 3.821278757725177, 'reg_alpha': 5.339764624690546, 'scale_pos_weight': 6.723400891593152, 'grow_policy': 'depthwise', 'max_leaves': 23, 'max_bin': 247, 'max_cat_to_onehot': 8, 'max_cat_threshold': 26, 'n_estimators': 614, 'sampling_method': 'uniform', 'subsample': 0.6152248546699406, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'eval_metric': 'auc', 'enable_categorical': True}
lightgbm_1k_cat_params = {'boosting_type': 'gbdt', 'num_leaves': 19, 'max_depth': 6, 'learning_rate': 0.013118621303382356, 'scale_pos_weight': 4.234357177755502, 'min_split_gain': 4.150101489297818, 'min_child_weight': 0.1439268407905771, 'min_child_samples': 47, 'colsample_bytree': 0.6376678823246574, 'reg_alpha': 0.29085902006459885, 'reg_lambda': 0.3198616601646805, 'colsample_bynode': 0.4286302115954977, 'min_data_per_group': 97, 'max_cat_threshold': 8, 'cat_l2': 30.200602897458936, 'cat_smooth': 0.014021044837121193, 'max_cat_to_onehot': 6, 'max_bin': 166, 'n_estimators': 372, 'subsample': 0.6440572461920292, 'subsample_freq': 4, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}
catboost_1k_cat_params = {'iterations': 433, 'learning_rate': 0.03554345124418867, 'depth': 4, 'l2_leaf_reg': 9.840089950178873, 'random_strength': 0.02805738171539979, 'rsm': 0.9135819869398158, 'border_count': 64, 'scale_pos_weight': 6.980459859158197, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 9.907283731079449, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_1k_cat_params = {'n_estimators': 737, 'learning_rate': 0.022671452328580047, 'minibatch_frac': 0.8921512142357929, 'col_sample': 0.4412224129149527, 'random_state': 42, 'verbose': False}
gbm_1k_cat_params = {'n_estimators': 579, 'learning_rate': 0.11372034355457401, 'max_depth': 3, 'subsample': 0.6608071029790215, 'min_samples_split': 44, 'min_samples_leaf': 36, 'min_weight_fraction_leaf': 0.13651164177932174, 'min_impurity_decrease': 0.91953008742171, 'max_features': 'log2', 'max_leaf_nodes': 22, 'ccp_alpha': 2.1238742124806448e-05, 'random_state': 42, 'verbose': 0}
histgbm_1k_cat_params = {'max_iter': 809, 'learning_rate': 0.005855268047892045, 'max_depth': 6, 'min_samples_leaf': 73, 'max_features': 0.6326799701936273, 'max_leaf_nodes': 31, 'l2_regularization': 0.00047402616662400567, 'max_bins': 252, 'random_state': 42, 'verbose': 0}
ngboost_1k_cat_tree_regressor_params = {'criterion': 'friedman_mse', 'max_depth': 7, 'min_samples_split': 61, 'min_samples_leaf': 66, 'min_weight_fraction_leaf': 0.1010094827157183, 'max_features': 'sqrt', 'ccp_alpha': 1.8200306725360187e-05, 'max_leaf_nodes': 25, 'min_impurity_decrease': 0.05036939294247465, 'random_state': 42}
ngboost_1k_cat_tree_regressor = DecisionTreeRegressor(**ngboost_1k_cat_tree_regressor_params)

models_tuned_1k_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_1k_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_1k_cat_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostClassifier(**catboost_1k_cat_params)),
    "NGBoost": (numeric_preprocessor, NGBClassifier(**ngboost_1k_cat_params, Dist=Bernoulli, Base=ngboost_1k_cat_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_1k_cat, GradientBoostingClassifier(**gbm_1k_cat_params)),
    "HistGBM": (numeric_preprocessor, HistGradientBoostingClassifier(**histgbm_1k_cat_params)),
    "Dummy - Most Frequent": (numeric_preprocessor, DummyClassifier(strategy="prior")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNClassifier())
}

In [11]:
xgboost_10k_cat_params = {'booster': 'gbtree', 'learning_rate': 0.002660634878062164, 'min_split_loss': 1.5564766508977637, 'max_depth': 3, 'min_child_weight': 0.002666795494419774, 'max_delta_step': 3.237106539752366, 'colsample_bytree': 0.6334129748685073, 'colsample_bylevel': 0.31232752047853835, 'colsample_bynode': 0.5663544691691175, 'reg_lambda': 0.866361380309008, 'reg_alpha': 0.34097661390173023, 'scale_pos_weight': 2.237901939773783, 'grow_policy': 'lossguide', 'max_leaves': 14, 'max_bin': 203, 'max_cat_to_onehot': 2, 'max_cat_threshold': 4, 'n_estimators': 1828, 'sampling_method': 'gradient_based', 'subsample': 0.21905666352297487, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'enable_categorical': True}
lightgbm_10k_cat_params = {'boosting_type': 'goss', 'num_leaves': 26, 'max_depth': 8, 'learning_rate': 0.0057452620856591865, 'scale_pos_weight': 2.5548028672067264, 'min_split_gain': 8.402184412779064, 'min_child_weight': 0.7889913973072552, 'min_child_samples': 67, 'colsample_bytree': 0.3322094184323836, 'reg_alpha': 0.009195021998688829, 'reg_lambda': 1.241109986317591, 'colsample_bynode': 0.3432164431126759, 'min_data_per_group': 13, 'max_cat_threshold': 3, 'cat_l2': 4.82194592281236, 'cat_smooth': 0.41553169506659904, 'max_cat_to_onehot': 1, 'max_bin': 66, 'n_estimators': 1022, 'top_rate': 0.07974625682135272, 'other_rate': 0.15560637534232696, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}
catboost_10k_cat_params = {'iterations': 1004, 'learning_rate': 0.007488378628866546, 'depth': 8, 'l2_leaf_reg': 0.008738906090086759, 'random_strength': 0.6403151618824401, 'rsm': 0.7659083035388369, 'border_count': 162, 'scale_pos_weight': 5.959581256258957, 'boosting_type': 'Plain', 'bootstrap_type': 'Bayesian', 'grow_policy': 'Lossguide', 'max_leaves': 17, 'min_data_in_leaf': 91, 'bagging_temperature': 8.58954612632319, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_10k_cat_params = {'n_estimators': 732, 'learning_rate': 0.004987425762907027, 'minibatch_frac': 0.7943140099019186, 'col_sample': 0.7946835959334093, 'random_state': 42, 'verbose': False}
gbm_10k_cat_params = {'n_estimators': 1588, 'learning_rate': 0.00549390058546573, 'max_depth': 3, 'subsample': 0.634594524243599, 'min_samples_split': 27, 'min_samples_leaf': 79, 'min_weight_fraction_leaf': 0.002500791243609685, 'min_impurity_decrease': 0.5395017070575661, 'max_features': 'log2', 'max_leaf_nodes': 8, 'ccp_alpha': 2.565584578912088e-05, 'random_state': 42, 'verbose': 0}
histgbm_10k_cat_params = {'max_iter': 1352, 'learning_rate': 0.0047997605390942095, 'max_depth': 4, 'min_samples_leaf': 53, 'max_features': 0.2772575122505353, 'max_leaf_nodes': 26, 'l2_regularization': 93.90576777776629, 'max_bins': 162, 'random_state': 42, 'verbose': 0}
ngboost_10k_cat_tree_regressor_params = {'criterion': 'friedman_mse', 'max_depth': 4, 'min_samples_split': 22, 'min_samples_leaf': 17, 'min_weight_fraction_leaf': 0.02731981862937974, 'max_features': 'log2', 'ccp_alpha': 2.0144676573632977e-05, 'max_leaf_nodes': 28, 'min_impurity_decrease': 0.18904335829713417, 'random_state': 42}
ngboost_10k_cat_tree_regressor = DecisionTreeRegressor(**ngboost_10k_cat_tree_regressor_params)

models_tuned_10k_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_10k_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_10k_cat_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostClassifier(**catboost_10k_cat_params)),
    "NGBoost": (numeric_preprocessor, NGBClassifier(**ngboost_10k_cat_params, Dist=Bernoulli, Base=ngboost_10k_cat_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_10k_cat, GradientBoostingClassifier(**gbm_10k_cat_params)),
    "HistGBM": (numeric_preprocessor, HistGradientBoostingClassifier(**histgbm_10k_cat_params)),
    "Dummy - Most Frequent": (numeric_preprocessor, DummyClassifier(strategy="prior")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNClassifier())
}

In [ ]:
xgboost_100k_cat_params = {'booster': 'gbtree', 'learning_rate': 0.0009878389008570133, 'min_split_loss': 1.6211471225466279, 'max_depth': 18, 'min_child_weight': 6.506529198811776, 'max_delta_step': 4.996596322660088, 'colsample_bytree': 0.5497212085853451, 'colsample_bylevel': 0.6402797842177399, 'colsample_bynode': 0.4945766078267192, 'reg_lambda': 3.901817763099591e-08, 'reg_alpha': 9.955075730252275, 'scale_pos_weight': 2.385489120852272, 'grow_policy': 'lossguide', 'max_leaves': 70, 'max_bin': 428, 'max_cat_to_onehot': 10, 'max_cat_threshold': 686, 'n_estimators': 9010, 'sampling_method': 'uniform', 'subsample': 0.34315380106756327, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'enable_categorical': True}
lightgbm_100k_cat_params = {'boosting_type': 'goss', 'num_leaves': 444, 'max_depth': 15, 'learning_rate': 0.0013490712742580085, 'scale_pos_weight': 2.529589934768151, 'min_split_gain': 3.4510864669147536, 'min_child_weight': 0.19239271728859866, 'min_child_samples': 234, 'colsample_bytree': 0.31324120666189326, 'reg_alpha': 0.6545836243558268, 'reg_lambda': 0.1069421692665621, 'colsample_bynode': 0.21348019642019067, 'min_data_per_group': 803, 'max_cat_threshold': 819, 'cat_l2': 0.00011204200863439848, 'cat_smooth': 1.5621591746591863, 'max_cat_to_onehot': 36, 'max_bin': 258, 'n_estimators': 8720, 'top_rate': 0.24249410683009498, 'other_rate': 0.42369427528780756, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}
catboost_100k_cat_params = {'iterations': 8940, 'learning_rate': 0.0025302116751832124, 'depth': 8, 'l2_leaf_reg': 98.54551873139987, 'random_strength': 0.030339567093011274, 'rsm': 0.8587177152995968, 'border_count': 185, 'scale_pos_weight': 6.177887840704495, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.7842935885515135, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_100k_cat_params = {'n_estimators': 9100, 'learning_rate': 0.001001318427544568, 'minibatch_frac': 0.6270741484049653, 'col_sample': 0.5670544735298176, 'random_state': 42, 'verbose': False}
gbm_100k_cat_params = {'n_estimators': 4750, 'learning_rate': 0.01648269774352038, 'max_depth': 16, 'subsample': 0.9090892401385037, 'min_samples_split': 443, 'min_samples_leaf': 75, 'min_weight_fraction_leaf': 0.09091249242263949, 'min_impurity_decrease': 0.6068460342082242, 'max_features': None, 'max_leaf_nodes': 82, 'ccp_alpha': 1.9782964485238465e-05, 'random_state': 42, 'verbose': 0}
histgbm_100k_cat_params = {'max_iter': 3930, 'learning_rate': 0.0028610506052507275, 'max_depth': 18, 'min_samples_leaf': 491, 'max_features': 0.25844075959233276, 'max_leaf_nodes': 273, 'l2_regularization': 1.2200390997850423, 'max_bins': 219, 'random_state': 42, 'verbose': 0}
ngboost_100k_cat_tree_regressor_params = {'criterion': 'friedman_mse', 'max_depth': 20, 'min_samples_split': 31, 'min_samples_leaf': 474, 'min_weight_fraction_leaf': 0.0021335723044131113, 'max_features': 'sqrt', 'ccp_alpha': 0.00028039529173976675, 'max_leaf_nodes': 300, 'min_impurity_decrease': 0.29132470764416524, 'random_state': 42}
ngboost_100k_cat_tree_regressor = DecisionTreeRegressor(**ngboost_100k_cat_tree_regressor_params)

models_tuned_100k_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_100k_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_100k_cat_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostClassifier(**catboost_100k_cat_params)),
    "NGBoost": (numeric_preprocessor, NGBClassifier(**ngboost_100k_cat_params, Dist=Bernoulli, Base=ngboost_100k_cat_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_100k_cat, GradientBoostingClassifier(**gbm_100k_cat_params)),
    "HistGBM": (numeric_preprocessor, HistGradientBoostingClassifier(**histgbm_100k_cat_params)),
    "Dummy - Most Frequent": (numeric_preprocessor, DummyClassifier(strategy="prior")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNClassifier())
}

In [ ]:
xgboost_full_cat_params = {'booster': 'gbtree', 'learning_rate': 0.0032927768001972434, 'min_split_loss': 8.210752775090127, 'max_depth': 7, 'min_child_weight': 0.15538873240656356, 'max_delta_step': 0.27402281636668574, 'colsample_bytree': 0.49786443299241995, 'colsample_bylevel': 0.5536721341549702, 'colsample_bynode': 0.6937873385849318, 'reg_lambda': 5.867840852070052, 'reg_alpha': 0.00036494609794361513, 'scale_pos_weight': 2.197402352020041, 'grow_policy': 'depthwise', 'max_leaves': 213, 'max_bin': 81, 'max_cat_to_onehot': 2, 'max_cat_threshold': 31, 'n_estimators': 9670, 'sampling_method': 'uniform', 'subsample': 0.693420186711674, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'binary:logistic', 'eval_metric': 'auc', 'enable_categorical': True}
lightgbm_full_cat_params = {'boosting_type': 'gbdt', 'num_leaves': 176, 'max_depth': 18, 'learning_rate': 0.0013263394883988124, 'scale_pos_weight': 5.720161074894982, 'min_split_gain': 7.015394333142449, 'min_child_weight': 0.7877329829167548, 'min_child_samples': 287, 'colsample_bytree': 0.6513068572006349, 'reg_alpha': 0.25347089784492205, 'reg_lambda': 5.51570221223381e-05, 'colsample_bynode': 0.33764943797808045, 'min_data_per_group': 550, 'max_cat_threshold': 187, 'cat_l2': 2.2848847335233635e-07, 'cat_smooth': 1.2867859076903958e-05, 'max_cat_to_onehot': 46, 'max_bin': 127, 'n_estimators': 9670, 'subsample': 0.7854117062367028, 'subsample_freq': 10, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'binary', 'metric': 'auc'}
catboost_full_cat_params = {'iterations': 8620, 'learning_rate': 0.005036874795758855, 'depth': 7, 'l2_leaf_reg': 28.934704951911797, 'random_strength': 0.5144245877609427, 'rsm': 0.5706570280810435, 'border_count': 105, 'scale_pos_weight': 7.201993563101878, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bernoulli', 'score_function': 'L2', 'subsample': 0.5675364012486861, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'Logloss', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_full_cat_params = {}
gbm_full_cat_params = {}
histgbm_full_cat_params = {'max_iter': 9910, 'learning_rate': 0.0009347843785877883, 'max_depth': 20, 'min_samples_leaf': 218, 'max_features': 0.31912233783575017, 'max_leaf_nodes': 437, 'l2_regularization': 55.83966409360982, 'max_bins': 231, 'random_state': 42, 'verbose': 0}
ngboost_full_cat_tree_regressor_params = {}
ngboost_full_cat_tree_regressor = DecisionTreeRegressor(**ngboost_full_cat_tree_regressor_params)

models_tuned_full_cat = {
    "XGBoost": (passthrough_preprocessor, XGBClassifier(**xgboost_full_cat_params)),
    "LightGBM": (passthrough_preprocessor, LGBMClassifier(**lightgbm_full_cat_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostClassifier(**catboost_full_cat_params)),
    #"NGBoost": (numeric_preprocessor, NGBClassifier(**ngboost_full_cat_params, Dist=Bernoulli, Base=ngboost_full_cat_tree_regressor)),
    #"GBM": (imputed_numeric_preprocessor_1k_cat, GradientBoostingClassifier(**gbm_full_cat_params)),
    "HistGBM": (numeric_preprocessor, HistGradientBoostingClassifier(**histgbm_full_cat_params)),
    "Dummy - Most Frequent": (numeric_preprocessor, DummyClassifier(strategy="prior")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNClassifier())
}

In [9]:
import os
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

def compare_models_cat(models_dict, train_X, train_y, test_X, test_y, dataset_size_name, tuning_status):
    out_dir_scores = "test_results/Classification/scores"
    out_dir_preds = "test_results/Classification/predictions"

    results_scores_test = []
    results_scores_train = []
    
    test_preds_df = pd.DataFrame(index=test_X.index)
    test_preds_df['true_y'] = test_y
    
    train_preds_df = pd.DataFrame(index=train_X.index)
    train_preds_df['true_y'] = train_y

    for model_name, (preprocessor, model) in models_dict.items():
        print(f"Evaluating {model_name} on {dataset_size_name} dataset ({tuning_status})...")
        pipe = Pipeline([("preprocessor", preprocessor), ("classifier", model)])

        if "TabPFN" in model_name:
            train_X_fit = train_X.tail(30000)
            train_y_fit = train_y.tail(30000)
            test_X_pred = test_X.head(20000)
            test_y_pred = test_y.head(20000)
            
            start_time_fit = time.time()
            pipe.fit(train_X_fit, train_y_fit)
            fit_time = time.time() - start_time_fit
            
            start_time_pred = time.time()
            test_pred_proba = pipe.predict_proba(test_X_pred)[:, 1]
            test_pred_time = time.time() - start_time_pred
            test_roc_auc = roc_auc_score(test_y_pred, test_pred_proba)
            test_pr_auc = average_precision_score(test_y_pred, test_pred_proba)
            
            start_time_pred_train = time.time()
            train_pred_proba = pipe.predict_proba(train_X_fit)[:, 1]
            train_pred_time = time.time() - start_time_pred_train
            train_roc_auc = roc_auc_score(train_y_fit, train_pred_proba)
            train_pr_auc = average_precision_score(train_y_fit, train_pred_proba)

            test_preds_df.loc[test_X_pred.index, model_name] = test_pred_proba
            train_preds_df.loc[train_X_fit.index, model_name] = train_pred_proba

        elif "CatBoost" in model_name:
            cb_train_X = train_X.copy()
            cb_test_X = test_X.copy()
            cat_cols = cb_train_X.select_dtypes(include=['object', 'category']).columns
            for col in cat_cols:
                cb_train_X[col] = cb_train_X[col].astype(object).fillna("Missing").astype(str)
                cb_test_X[col] = cb_test_X[col].astype(object).fillna("Missing").astype(str)
            
            start_time_fit = time.time()
            pipe.fit(cb_train_X, train_y)
            fit_time = time.time() - start_time_fit

            start_time_pred = time.time()
            test_pred_proba = pipe.predict_proba(cb_test_X)[:, 1]
            test_pred_time = time.time() - start_time_pred
            test_roc_auc = roc_auc_score(test_y, test_pred_proba)
            test_pr_auc = average_precision_score(test_y, test_pred_proba)

            start_time_pred_train = time.time()
            train_pred_proba = pipe.predict_proba(cb_train_X)[:, 1]
            train_pred_time = time.time() - start_time_pred_train
            train_roc_auc = roc_auc_score(train_y, train_pred_proba)
            train_pr_auc = average_precision_score(train_y, train_pred_proba)
            
            test_preds_df[model_name] = test_pred_proba
            train_preds_df[model_name] = train_pred_proba

        else:
            start_time_fit = time.time()
            if "NGBoost" in model_name:
                weights = compute_sample_weight(class_weight="balanced", y=train_y)
                pipe.fit(train_X, train_y, classifier__sample_weight=weights)
            else:
                pipe.fit(train_X, train_y)
            fit_time = time.time() - start_time_fit
            
            start_time_pred = time.time()
            test_pred_proba = pipe.predict_proba(test_X)[:, 1]
            test_pred_time = time.time() - start_time_pred
            test_roc_auc = roc_auc_score(test_y, test_pred_proba)
            test_pr_auc = average_precision_score(test_y, test_pred_proba)

            start_time_pred_train = time.time()
            train_pred_proba = pipe.predict_proba(train_X)[:, 1]
            train_pred_time = time.time() - start_time_pred_train
            train_roc_auc = roc_auc_score(train_y, train_pred_proba)
            train_pr_auc = average_precision_score(train_y, train_pred_proba)

            test_preds_df[model_name] = test_pred_proba
            train_preds_df[model_name] = train_pred_proba

        results_scores_test.append({
            "Dataset": dataset_size_name, "Model": model_name, "ROC AUC": test_roc_auc, "PR AUC": test_pr_auc,
            "Fit Time (s)": fit_time, "Pred Time (s)": test_pred_time
        })
        results_scores_train.append({
            "Dataset": dataset_size_name, "Model": model_name, "ROC AUC": train_roc_auc, "PR AUC": train_pr_auc,
            "Fit Time (s)": fit_time, "Pred Time (s)": train_pred_time
        })

    df_scores_test = pd.DataFrame(results_scores_test)
    df_scores_train = pd.DataFrame(results_scores_train)

    test_preds_path = os.path.join(out_dir_preds, f"test_preds_{dataset_size_name}_{tuning_status}.parquet")
    train_preds_path = os.path.join(out_dir_preds, f"train_preds_{dataset_size_name}_{tuning_status}.parquet")
    test_scores_path = os.path.join(out_dir_scores, f"test_scores_{dataset_size_name}_{tuning_status}.csv")
    train_scores_path = os.path.join(out_dir_scores, f"train_scores_{dataset_size_name}_{tuning_status}.csv")

    test_preds_df.to_parquet(test_preds_path)
    train_preds_df.to_parquet(train_preds_path)
    df_scores_test.to_csv(test_scores_path, index=False)
    df_scores_train.to_csv(train_scores_path, index=False)

    print(f"Finished evaluating {dataset_size_name} dataset ({tuning_status}). Results saved to {out_dir_scores}/ and {out_dir_preds}/.")

    return df_scores_test, df_scores_train, test_preds_df, train_preds_df

In [13]:
compare_models_cat(models_no_tuning_cat, train_1k_X, train_1k_y_cat, test_full_X, test_full_y_cat, "1k", "not_tuned")

Evaluating XGBoost on 1k dataset (not_tuned)...
Evaluating LightGBM on 1k dataset (not_tuned)...
Evaluating CatBoost on 1k dataset (not_tuned)...
Evaluating NGBoost on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating Dummy - Most Frequent on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Finished evaluating 1k dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


(  Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0      1k                XGBoost  0.650179  0.322478      0.160058   
 1      1k               LightGBM  0.656751  0.325501      0.399002   
 2      1k               CatBoost  0.678580  0.347285      2.509436   
 3      1k                NGBoost  0.675680  0.335933      4.311730   
 4      1k                    GBM  0.659474  0.324237      0.745799   
 5      1k                HistGBM  0.658171  0.323029      0.479424   
 6      1k  Dummy - Most Frequent  0.500000  0.214253      0.005258   
 
    Pred Time (s)  
 0       0.138293  
 1       0.383374  
 2       0.273622  
 3      10.899444  
 4       1.248871  
 5       0.746848  
 6       0.291573  ,
   Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0      1k                XGBoost  1.000000  1.000000      0.160058   
 1      1k               LightGBM  1.000000  1.000000      0.399002   
 2      1k               CatBoost  0.998963  0.996943   

In [14]:
compare_models_cat(models_no_tuning_cat, train_10k_X, train_10k_y_cat, test_full_X, test_full_y_cat, "10k", "not_tuned")

Evaluating XGBoost on 10k dataset (not_tuned)...
Evaluating LightGBM on 10k dataset (not_tuned)...
Evaluating CatBoost on 10k dataset (not_tuned)...
Evaluating NGBoost on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating Dummy - Most Frequent on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Finished evaluating 10k dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


(  Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0     10k                XGBoost  0.665033  0.336802      0.335519   
 1     10k               LightGBM  0.696342  0.368995      0.482216   
 2     10k               CatBoost  0.711886  0.388682      6.715328   
 3     10k                NGBoost  0.705868  0.383333     39.915622   
 4     10k                    GBM  0.705263  0.379836      6.877951   
 5     10k                HistGBM  0.696484  0.368486      0.621236   
 6     10k  Dummy - Most Frequent  0.500000  0.214253      0.014800   
 
    Pred Time (s)  
 0       0.128767  
 1       0.248421  
 2       0.267471  
 3      10.698462  
 4       1.231220  
 5       0.588356  
 6       0.291055  ,
   Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0     10k                XGBoost  0.999812  0.999457      0.335519   
 1     10k               LightGBM  0.979322  0.946979      0.482216   
 2     10k               CatBoost  0.927036  0.849076   

In [15]:
compare_models_cat(models_no_tuning_cat, train_100k_X, train_100k_y_cat, test_full_X, test_full_y_cat, "100k", "not_tuned")

Evaluating XGBoost on 100k dataset (not_tuned)...
Evaluating LightGBM on 100k dataset (not_tuned)...
Evaluating CatBoost on 100k dataset (not_tuned)...
Evaluating NGBoost on 100k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 100k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 100k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating Dummy - Most Frequent on 100k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Finished evaluating 100k dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


(  Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0    100k                XGBoost  0.706904  0.386137      1.190197   
 1    100k               LightGBM  0.724336  0.408414      1.028174   
 2    100k               CatBoost  0.729006  0.412856     30.238377   
 3    100k                NGBoost  0.716783  0.398961    404.685028   
 4    100k                    GBM  0.719454  0.402528     68.947165   
 5    100k                HistGBM  0.723687  0.407550      1.851496   
 6    100k  Dummy - Most Frequent  0.500000  0.214253      0.107103   
 
    Pred Time (s)  
 0       0.148327  
 1       0.274704  
 2       0.267619  
 3      10.911484  
 4       1.237453  
 5       0.552437  
 6       0.238423  ,
   Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0    100k                XGBoost  0.878127  0.739829      1.190197   
 1    100k               LightGBM  0.780007  0.555470      1.028174   
 2    100k               CatBoost  0.813665  0.635952   

In [16]:
compare_models_cat(models_no_tuning_cat, train_full_X, train_full_y_cat, test_full_X, test_full_y_cat, "full", "not_tuned")

Evaluating XGBoost on full dataset (not_tuned)...
Evaluating LightGBM on full dataset (not_tuned)...
Evaluating CatBoost on full dataset (not_tuned)...
Evaluating NGBoost on full dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on full dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on full dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating Dummy - Most Frequent on full dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Finished evaluating full dataset (not_tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


(  Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0    full                XGBoost  0.721658  0.403980      9.781763   
 1    full               LightGBM  0.721926  0.405490      5.949047   
 2    full               CatBoost  0.725213  0.408408    296.543367   
 3    full                NGBoost  0.706432  0.382604   5430.830452   
 4    full                    GBM  0.709641  0.388374    732.561134   
 5    full                HistGBM  0.720956  0.404598     19.682994   
 6    full  Dummy - Most Frequent  0.500000  0.214253      1.157760   
 
    Pred Time (s)  
 0       0.142847  
 1       0.413401  
 2       0.340449  
 3      11.374909  
 4       1.372440  
 5       0.677115  
 6       0.238587  ,
   Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0    full                XGBoost  0.768326  0.464807      9.781763   
 1    full               LightGBM  0.742379  0.418292      5.949047   
 2    full               CatBoost  0.767558  0.476952   

In [11]:
compare_models_cat(models_tuned_1k_cat, train_1k_X, train_1k_y_cat, test_full_X, test_full_y_cat, "1k", "tuned")

Evaluating XGBoost on 1k dataset (tuned)...
Evaluating LightGBM on 1k dataset (tuned)...
Evaluating CatBoost on 1k dataset (tuned)...
Evaluating NGBoost on 1k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 1k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 1k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating Dummy - Most Frequent on 1k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Finished evaluating 1k dataset (tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


(  Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0      1k                XGBoost  0.668767  0.337649      0.503774   
 1      1k               LightGBM  0.683716  0.352338      0.350462   
 2      1k               CatBoost  0.679436  0.349067      0.905078   
 3      1k                NGBoost  0.681545  0.350068      1.292962   
 4      1k                    GBM  0.683122  0.355808      0.289137   
 5      1k                HistGBM  0.676844  0.345438      1.107735   
 6      1k  Dummy - Most Frequent  0.500000  0.214253      0.006008   
 
    Pred Time (s)  
 0       0.277555  
 1       0.624756  
 2       0.202887  
 3       9.746179  
 4       1.326566  
 5       2.052047  
 6       0.295319  ,
   Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0      1k                XGBoost  0.996854  0.987822      0.503774   
 1      1k               LightGBM  0.923486  0.789284      0.350462   
 2      1k               CatBoost  0.961538  0.889620   

In [12]:
compare_models_cat(models_tuned_10k_cat, train_10k_X, train_10k_y_cat, test_full_X, test_full_y_cat, "10k", "tuned")

Evaluating XGBoost on 10k dataset (tuned)...
Evaluating LightGBM on 10k dataset (tuned)...
Evaluating CatBoost on 10k dataset (tuned)...
Evaluating NGBoost on 10k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 10k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 10k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating Dummy - Most Frequent on 10k dataset (tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Finished evaluating 10k dataset (tuned). Results saved to test_results/Classification/scores/ and test_results/Classification/predictions/.


(  Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0     10k                XGBoost  0.711726  0.391639      2.930773   
 1     10k               LightGBM  0.714818  0.394123      3.238024   
 2     10k               CatBoost  0.706508  0.383853      7.932646   
 3     10k                NGBoost  0.711257  0.390929      9.017442   
 4     10k                    GBM  0.710844  0.389307      6.112199   
 5     10k                HistGBM  0.709828  0.387400      3.390730   
 6     10k  Dummy - Most Frequent  0.500000  0.214253      0.014822   
 
    Pred Time (s)  
 0       0.722709  
 1       3.302255  
 2       0.742003  
 3      13.972501  
 4       6.858024  
 5       2.942415  
 6       0.290964  ,
   Dataset                  Model   ROC AUC    PR AUC  Fit Time (s)  \
 0     10k                XGBoost  0.742388  0.455275      2.930773   
 1     10k               LightGBM  0.810074  0.552932      3.238024   
 2     10k               CatBoost  0.731073  0.436232   

In [ ]:
compare_models_cat(models_tuned_100k_cat, train_100k_X, train_100k_y_cat, test_full_X, test_full_y_cat, "100k", "tuned")

In [ ]:
compare_models_cat(models_tuned_full_cat, train_full_X, train_full_y_cat, test_full_X, test_full_y_cat, "full", "tuned")

## Regression

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor_1k_reg = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="mean")
).set_output(transform="pandas")

imputed_numeric_preprocessor_10k_reg = make_pipeline(
    numeric_preprocessor,
    IterativeImputer(max_iter=10, random_state=42)
).set_output(transform="pandas")

imputed_numeric_preprocessor_100k_reg = make_pipeline(
    numeric_preprocessor,
    IterativeImputer(max_iter=10, random_state=42)
).set_output(transform="pandas")

passthrough_preprocessor = "passthrough"

Categorical columns:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from ngboost import NGBRegressor
from tabpfn_client import TabPFNRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor as HistGBMRegressor
from pgbm.sklearn import HistGradientBoostingRegressor as PGBMRegressor
from ngboost.distns import Normal
from sklearn.tree import DecisionTreeRegressor


models_no_tuning_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(random_state=42, enable_categorical=True)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(random_state=42)),
    "CatBoost": (passthrough_preprocessor, CatBoostRegressor(random_state=42, verbose=0, cat_features=cols_to_nominal_cat)),
    "NGBoost": (numeric_preprocessor, NGBRegressor(random_state=42, verbose=False)),
    "GBM": (imputed_numeric_preprocessor_1k_reg, GradientBoostingRegressor(random_state=42)),
    "HistGBM": (numeric_preprocessor, HistGBMRegressor(random_state=42)),
    "PGBM": (numeric_preprocessor, PGBMRegressor(random_state=42)),
    "Dummy - Mean": (numeric_preprocessor, DummyRegressor(strategy="mean")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNRegressor())
}

In [ ]:
xgboost_1k_reg_params = {'booster': 'gbtree', 'learning_rate': 0.006592924714393806, 'min_split_loss': 0.3399903461254572, 'max_depth': 7, 'min_child_weight': 3.6857527823293634, 'max_delta_step': 3.928223239451256, 'colsample_bytree': 0.381061252694662, 'colsample_bylevel': 0.3290330497906662, 'colsample_bynode': 0.3640630952340392, 'reg_lambda': 0.01062303202535693, 'reg_alpha': 0.03812077743370024, 'grow_policy': 'lossguide', 'max_leaves': 20, 'max_bin': 169, 'max_cat_to_onehot': 18, 'max_cat_threshold': 11, 'n_estimators': 537, 'sampling_method': 'uniform', 'subsample': 0.5636865373558122, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}
lightgbm_1k_reg_params = {'boosting_type': 'gbdt', 'num_leaves': 13, 'max_depth': 3, 'learning_rate': 0.02169809132057075, 'min_split_gain': 0.027389027279699385, 'min_child_weight': 0.6899288118460188, 'min_child_samples': 42, 'colsample_bytree': 0.37238284939773186, 'reg_alpha': 0.09371206330869274, 'reg_lambda': 0.019391247189888383, 'colsample_bynode': 0.30902821981225653, 'min_data_per_group': 33, 'max_cat_threshold': 29, 'cat_l2': 0.041250555077113255, 'cat_smooth': 28.95327386652521, 'max_cat_to_onehot': 6, 'max_bin': 161, 'n_estimators': 144, 'subsample': 0.8693945506758759, 'subsample_freq': 5, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}
catboost_1k_reg_params = {'iterations': 976, 'learning_rate': 0.00976818825318037, 'depth': 4, 'l2_leaf_reg': 0.027265370477102587, 'random_strength': 0.7031697546586838, 'rsm': 0.8385396695909164, 'border_count': 201, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'Cosine', 'bagging_temperature': 9.530476491737797, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_1k_reg_params = {'n_estimators': 263, 'learning_rate': 0.015377425834686074, 'minibatch_frac': 0.663571492569765, 'col_sample': 0.5226236509136446, 'random_state': 42, 'verbose': False}
gbm_1k_reg_params = {'n_estimators': 971, 'learning_rate': 0.01388668367558402, 'max_depth': 8, 'subsample': 0.30872672854426325, 'min_samples_split': 43, 'min_samples_leaf': 28, 'min_weight_fraction_leaf': 0.3945207700562416, 'min_impurity_decrease': 0.16016521565758096, 'max_features': 'sqrt', 'max_leaf_nodes': 6, 'ccp_alpha': 3.910485014559049e-05, 'random_state': 42, 'verbose': 0}
histgbm_1k_reg_params = {'max_iter': 734, 'learning_rate': 0.006736563678671684, 'max_depth': 2, 'min_samples_leaf': 52, 'max_features': 0.203174979863919, 'max_leaf_nodes': 6, 'l2_regularization': 0.0063066439175882, 'max_bins': 124, 'random_state': 42, 'verbose': 0}
pgbm_1k_reg_params = {'max_iter': 266, 'learning_rate': 0.0057633608637110215, 'max_depth': 2, 'min_samples_leaf': 80, 'max_leaf_nodes': 24, 'l2_regularization': 3.757726025937608e-05, 'max_bins': 106, 'tree_correlation': 0.00017249809953907554, 'distribution': 'laplace', 'random_state': 42, 'verbose': 0, 'with_variance': True}
ngboost_1k_reg_tree_regressor_params = {'criterion': 'friedman_mse', 'max_depth': 8, 'min_samples_split': 25, 'min_samples_leaf': 29, 'min_weight_fraction_leaf': 0.10108332219836896, 'max_features': 'log2', 'ccp_alpha': 0.00011946599313498593, 'max_leaf_nodes': 6, 'min_impurity_decrease': 0.289818830272591, 'random_state': 42}
ngboost_1k_reg_tree_regressor = DecisionTreeRegressor(**ngboost_1k_reg_tree_regressor_params)

models_tuned_1k_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_1k_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_1k_reg_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostRegressor(**catboost_1k_reg_params)),
    "NGBoost": (numeric_preprocessor, NGBRegressor(**ngboost_1k_reg_params, Dist=Normal, Base=ngboost_1k_reg_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_1k_reg, GradientBoostingRegressor(**gbm_1k_reg_params)),
    "HistGBM": (numeric_preprocessor, HistGBMRegressor(**histgbm_1k_reg_params)),
    "PGBM": (numeric_preprocessor, PGBMRegressor(**pgbm_1k_reg_params)),
    "Dummy - Mean": (numeric_preprocessor, DummyRegressor(strategy="mean")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNRegressor())
}

In [ ]:
xgboost_10k_reg_params = {'booster': 'gbtree', 'learning_rate': 0.0034649471654673372, 'min_split_loss': 1.2225056392066438, 'max_depth': 4, 'min_child_weight': 0.044868703229338794, 'max_delta_step': 7.322896593525407, 'colsample_bytree': 0.664513006915934, 'colsample_bylevel': 0.30183598693553104, 'colsample_bynode': 0.6199112663715858, 'reg_lambda': 0.022303464542099465, 'reg_alpha': 8.45346616493749, 'grow_policy': 'depthwise', 'max_leaves': 31, 'max_bin': 69, 'max_cat_to_onehot': 3, 'max_cat_threshold': 32, 'n_estimators': 1990, 'sampling_method': 'gradient_based', 'subsample': 0.19636351824285336, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}
lightgbm_10k_reg_params = {'boosting_type': 'goss', 'num_leaves': 28, 'max_depth': 3, 'learning_rate': 0.00573759926773727, 'min_split_gain': 0.041494825179870576, 'min_child_weight': 0.012349230729538438, 'min_child_samples': 81, 'colsample_bytree': 0.5283693462952471, 'reg_alpha': 9.158641225780748, 'reg_lambda': 0.004093641172128938, 'colsample_bynode': 0.6725807827106883, 'min_data_per_group': 91, 'max_cat_threshold': 6, 'cat_l2': 1.5908743669801766, 'cat_smooth': 0.14729665520231705, 'max_cat_to_onehot': 11, 'max_bin': 156, 'n_estimators': 1494, 'top_rate': 0.1092716955430614, 'other_rate': 0.777449860751789, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}
catboost_10k_reg_params = {'iterations': 1088, 'learning_rate': 0.013825345653490797, 'depth': 5, 'l2_leaf_reg': 0.5148645656737443, 'random_strength': 0.1893828349911231, 'rsm': 0.7999433598790255, 'border_count': 194, 'boosting_type': 'Ordered', 'bootstrap_type': 'Bayesian', 'score_function': 'L2', 'bagging_temperature': 8.170909952414544, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_10k_reg_params = {'n_estimators': 1428, 'learning_rate': 0.005743155031743333, 'minibatch_frac': 0.5442236810583654, 'col_sample': 0.7169685029737407, 'random_state': 42, 'verbose': False}
gbm_10k_reg_params = {'n_estimators': 1876, 'learning_rate': 0.022112077171996453, 'max_depth': 3, 'subsample': 0.4440702904540561, 'min_samples_split': 16, 'min_samples_leaf': 40, 'min_weight_fraction_leaf': 0.07834199132955552, 'min_impurity_decrease': 0.7616676770612814, 'max_features': 'log2', 'max_leaf_nodes': 22, 'ccp_alpha': 1.5744809085067783e-05, 'random_state': 42, 'verbose': 0}
histgbm_10k_reg_params = {'max_iter': 878, 'learning_rate': 0.004906344452687288, 'max_depth': 5, 'min_samples_leaf': 23, 'max_features': 0.3270420473977633, 'max_leaf_nodes': 5, 'l2_regularization': 0.0003377040719829344, 'max_bins': 71, 'random_state': 42, 'verbose': 0}
pgbm_10k_reg_params = {'max_iter': 670, 'learning_rate': 0.0046421620560510345, 'max_depth': 6, 'min_samples_leaf': 26, 'max_leaf_nodes': 8, 'l2_regularization': 86.43849217392876, 'max_bins': 149, 'tree_correlation': 0.030672618536641574, 'distribution': 'laplace', 'random_state': 42, 'verbose': 0, 'with_variance': True}
ngboost_10k_reg_tree_regressor_params = {'criterion': 'friedman_mse', 'max_depth': 8, 'min_samples_split': 38, 'min_samples_leaf': 41, 'min_weight_fraction_leaf': 0.0003213921303004477, 'max_features': 'log2', 'ccp_alpha': 1.8549133165544335e-05, 'max_leaf_nodes': 17, 'min_impurity_decrease': 0.38252906628451194, 'random_state': 42}
ngboost_10k_reg_tree_regressor = DecisionTreeRegressor(**ngboost_10k_reg_tree_regressor_params)

models_tuned_10k_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_10k_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_10k_reg_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostRegressor(**catboost_10k_reg_params)),
    "NGBoost": (numeric_preprocessor, NGBRegressor(**ngboost_10k_reg_params, Dist=Normal, Base=ngboost_10k_reg_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_10k_reg, GradientBoostingRegressor(**gbm_10k_reg_params)),
    "HistGBM": (numeric_preprocessor, HistGBMRegressor(**histgbm_10k_reg_params)),
    "PGBM": (numeric_preprocessor, PGBMRegressor(**pgbm_10k_reg_params)),
    "Dummy - Mean": (numeric_preprocessor, DummyRegressor(strategy="mean")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNRegressor())
}

In [ ]:
xgboost_100k_reg_params = {'booster': 'gbtree', 'learning_rate': 0.00358874054592766, 'min_split_loss': 0.30665873968801743, 'max_depth': 9, 'min_child_weight': 0.058208746224353215, 'max_delta_step': 6.600824779885242, 'colsample_bytree': 0.31452203665302875, 'colsample_bylevel': 0.7845215630136946, 'colsample_bynode': 0.45580656447386975, 'reg_lambda': 1.4591079946865122, 'reg_alpha': 7.300918108293281, 'grow_policy': 'depthwise', 'max_leaves': 311, 'max_bin': 270, 'max_cat_to_onehot': 25, 'max_cat_threshold': 578, 'n_estimators': 9240, 'sampling_method': 'uniform', 'subsample': 0.892493539712304, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}
lightgbm_100k_reg_params = {'boosting_type': 'gbdt', 'num_leaves': 321, 'max_depth': 7, 'learning_rate': 0.0014957375416499211, 'min_split_gain': 0.15169683434030254, 'min_child_weight': 0.0043443738122844215, 'min_child_samples': 129, 'colsample_bytree': 0.5703676576888828, 'reg_alpha': 0.0009598013297060651, 'reg_lambda': 1.512993041670286e-06, 'colsample_bynode': 0.40283945018201495, 'min_data_per_group': 572, 'max_cat_threshold': 557, 'cat_l2': 0.1829638573103202, 'cat_smooth': 72.95247318237497, 'max_cat_to_onehot': 51, 'max_bin': 341, 'n_estimators': 6740, 'subsample': 0.998415655290472, 'subsample_freq': 7, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}
catboost_100k_reg_params = {'iterations': 9930, 'learning_rate': 0.001087680526009645, 'depth': 9, 'l2_leaf_reg': 25.36922452973143, 'random_strength': 1.1801448166458588, 'rsm': 0.7742077921478168, 'border_count': 217, 'boosting_type': 'Plain', 'bootstrap_type': 'MVS', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 299, 'score_function': 'L2', 'subsample': 0.269182776220681, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_100k_reg_params = {'n_estimators': 3540, 'learning_rate': 0.0031005975042584922, 'minibatch_frac': 0.5337656312465495, 'col_sample': 0.7957079748440611, 'random_state': 42, 'verbose': False}
gbm_100k_reg_params = {'n_estimators': 6020, 'learning_rate': 0.004590009050693113, 'max_depth': 4, 'subsample': 0.3702702077447556, 'min_samples_split': 343, 'min_samples_leaf': 242, 'min_weight_fraction_leaf': 0.020346710861258076, 'min_impurity_decrease': 0.8240878364564256, 'max_features': None, 'max_leaf_nodes': 320, 'ccp_alpha': 3.276614763080726e-05, 'random_state': 42, 'verbose': 0}
histgbm_100k_reg_params = {'max_iter': 4800, 'learning_rate': 0.001430411566426403, 'max_depth': 8, 'min_samples_leaf': 259, 'max_features': 0.4398281267806546, 'max_leaf_nodes': 43, 'l2_regularization': 0.00024044149018689098, 'max_bins': 147, 'random_state': 42, 'verbose': 0}
pgbm_100k_reg_params = {'max_iter': 9490, 'learning_rate': 0.0069166877386120885, 'max_depth': 10, 'min_samples_leaf': 327, 'max_leaf_nodes': 24, 'l2_regularization': 14.804147651872958, 'max_bins': 169, 'tree_correlation': 0.012392279453673792, 'distribution': 'laplace', 'random_state': 42, 'verbose': 0, 'with_variance': True}
ngboost_100k_reg_tree_regressor_params = {'criterion': 'friedman_mse', 'max_depth': 17, 'min_samples_split': 35, 'min_samples_leaf': 371, 'min_weight_fraction_leaf': 0.003373989905557241, 'max_features': 'sqrt', 'ccp_alpha': 2.1975874211460698e-05, 'max_leaf_nodes': 30, 'min_impurity_decrease': 0.035785241864927805, 'random_state': 42}
ngboost_100k_reg_tree_regressor = DecisionTreeRegressor(**ngboost_100k_reg_tree_regressor_params)

models_tuned_100k_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_100k_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_100k_reg_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostRegressor(**catboost_100k_reg_params)),
    "NGBoost": (numeric_preprocessor, NGBRegressor(**ngboost_100k_reg_params, Dist=Normal, Base=ngboost_100k_reg_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_100k_reg, GradientBoostingRegressor(**gbm_100k_reg_params)),
    "HistGBM": (numeric_preprocessor, HistGBMRegressor(**histgbm_100k_reg_params)),
    "PGBM": (numeric_preprocessor, PGBMRegressor(**pgbm_100k_reg_params)),
    "Dummy - Mean": (numeric_preprocessor, DummyRegressor(strategy="mean")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNRegressor())
}

In [ ]:
xgboost_full_reg_params = {'booster': 'gbtree', 'learning_rate': 0.0015361738364973562, 'min_split_loss': 0.019158435834981884, 'max_depth': 12, 'min_child_weight': 5.46291036872166, 'max_delta_step': 4.8592833061852465, 'colsample_bytree': 0.78840555707391, 'colsample_bylevel': 0.3400841955942318, 'colsample_bynode': 0.7797771602511868, 'reg_lambda': 0.001536038127856472, 'reg_alpha': 0.00444179665468952, 'grow_policy': 'lossguide', 'max_leaves': 303, 'max_bin': 352, 'max_cat_to_onehot': 18, 'max_cat_threshold': 668, 'n_estimators': 6800, 'sampling_method': 'uniform', 'subsample': 0.9347372849225181, 'random_state': 42, 'n_jobs': -1, 'verbosity': 0, 'tree_method': 'hist', 'objective': 'reg:squarederror', 'enable_categorical': True}
lightgbm_full_reg_params = {'boosting_type': 'gbdt', 'num_leaves': 367, 'max_depth': 9, 'learning_rate': 0.0037106925990830716, 'min_split_gain': 0.028541720097881224, 'min_child_weight': 3.114973690234082e-05, 'min_child_samples': 449, 'colsample_bytree': 0.9583470008702835, 'reg_alpha': 4.042104970937467, 'reg_lambda': 4.013246511788023e-08, 'colsample_bynode': 0.544229805808371, 'min_data_per_group': 745, 'max_cat_threshold': 240, 'cat_l2': 1.4174693320549176e-05, 'cat_smooth': 0.006490614096342806, 'max_cat_to_onehot': 4, 'max_bin': 97, 'n_estimators': 5940, 'subsample': 0.9848817124796165, 'subsample_freq': 7, 'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'objective': 'regression', 'metric': 'rmse'}
catboost_full_reg_params = {'iterations': 8780, 'learning_rate': 0.0019516220841303295, 'depth': 9, 'l2_leaf_reg': 2.9869534553308876e-07, 'random_strength': 0.028533049262069288, 'rsm': 0.41630399524229533, 'border_count': 167, 'boosting_type': 'Plain', 'bootstrap_type': 'Bernoulli', 'grow_policy': 'Depthwise', 'min_data_in_leaf': 482, 'score_function': 'L2', 'subsample': 0.8907108810335598, 'random_state': 42, 'thread_count': 7, 'verbose': False, 'objective': 'RMSE', 'cat_features': ['home_ownership', 'verification_status', 'purpose', 'addr_state', 'initial_list_status', 'application_type', 'disbursement_method']}
ngboost_full_reg_params = {}
gbm_full_reg_params = {}
histgbm_full_reg_params = {'max_iter': 6790, 'learning_rate': 0.0018532518685825527, 'max_depth': 15, 'min_samples_leaf': 362, 'max_features': 0.4243152639220175, 'max_leaf_nodes': 206, 'l2_regularization': 0.06879760344829619, 'max_bins': 213, 'random_state': 42, 'verbose': 0}
pgbm_full_reg_params = {'max_iter': 5110, 'learning_rate': 0.001967232115519223, 'max_depth': 14, 'min_samples_leaf': 296, 'max_leaf_nodes': 192, 'l2_regularization': 0.38777665330714267, 'max_bins': 185, 'tree_correlation': 0.0010571351786840086, 'distribution': 'logistic', 'random_state': 42, 'verbose': 0, 'with_variance': True}
ngboost_full_reg_tree_regressor_params = {}
ngboost_full_reg_tree_regressor = DecisionTreeRegressor(**ngboost_full_reg_tree_regressor_params)

models_tuned_full_reg = {
    "XGBoost": (passthrough_preprocessor, XGBRegressor(**xgboost_full_reg_params)),
    "LightGBM": (passthrough_preprocessor, LGBMRegressor(**lightgbm_full_reg_params)),
    "CatBoost": (passthrough_preprocessor, CatBoostRegressor(**catboost_full_reg_params)),
    #"NGBoost": (numeric_preprocessor, NGBRegressor(**ngboost_full_reg_params, Dist=Normal, Base=ngboost_full_reg_tree_regressor)),
    "GBM": (imputed_numeric_preprocessor_1k_reg, GradientBoostingRegressor(**gbm_full_reg_params)),
    "HistGBM": (numeric_preprocessor, HistGBMRegressor(**histgbm_full_reg_params)),
    #"PGBM": (numeric_preprocessor, PGBMRegressor(**pgbm_full_reg_params)),
    "Dummy - Mean": (numeric_preprocessor, DummyRegressor(strategy="mean")),
    #"TabPFN API": (passthrough_preprocessor, TabPFNRegressor())
}

In [16]:
import os
import time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, 
    r2_score, 
    mean_absolute_error, 
    mean_absolute_percentage_error
)

def compare_models_reg(models_dict, train_X, train_y, test_X, test_y, dataset_size_name, tuning_status):
    out_dir_scores = "test_results/Regression/scores"
    out_dir_preds = "test_results/Regression/predictions"

    results_scores_test = []
    results_scores_train = []
    
    test_preds_df = pd.DataFrame(index=test_X.index)
    test_preds_df['true_y'] = test_y
    
    train_preds_df = pd.DataFrame(index=train_X.index)
    train_preds_df['true_y'] = train_y

    for model_name, (preprocessor, model) in models_dict.items():
        print(f"Evaluating {model_name} on {dataset_size_name} dataset ({tuning_status})...")
        pipe = Pipeline([("preprocessor", preprocessor), ("regressor", model)])

        if "TabPFN" in model_name:
            train_X_fit = train_X.tail(30000)
            train_y_fit = train_y.tail(30000)
            test_X_pred = test_X.head(20000)
            test_y_pred = test_y.head(20000)
            
            start_time_fit = time.time()
            pipe.fit(train_X_fit, train_y_fit)
            fit_time = time.time() - start_time_fit
            
            start_time_pred = time.time()
            test_pred = pipe.predict(test_X_pred)  # Změna na predict
            test_pred_time = time.time() - start_time_pred
            
            start_time_pred_train = time.time()
            train_pred = pipe.predict(train_X_fit) # Změna na predict
            train_pred_time = time.time() - start_time_pred_train

            test_rmse = np.sqrt(mean_squared_error(test_y_pred, test_pred))
            test_r2 = r2_score(test_y_pred, test_pred)
            test_mae = mean_absolute_error(test_y_pred, test_pred)
            test_mape = mean_absolute_percentage_error(test_y_pred, test_pred)
            
            train_rmse = np.sqrt(mean_squared_error(train_y_fit, train_pred))
            train_r2 = r2_score(train_y_fit, train_pred)
            train_mae = mean_absolute_error(train_y_fit, train_pred)
            train_mape = mean_absolute_percentage_error(train_y_fit, train_pred)

            test_preds_df.loc[test_X_pred.index, model_name] = test_pred
            train_preds_df.loc[train_X_fit.index, model_name] = train_pred

        elif "CatBoost" in model_name:
            cb_train_X = train_X.copy()
            cb_test_X = test_X.copy()
            cat_cols = cb_train_X.select_dtypes(include=['object', 'category']).columns
            for col in cat_cols:
                cb_train_X[col] = cb_train_X[col].astype(object).fillna("Missing").astype(str)
                cb_test_X[col] = cb_test_X[col].astype(object).fillna("Missing").astype(str)
            
            start_time_fit = time.time()
            pipe.fit(cb_train_X, train_y)
            fit_time = time.time() - start_time_fit

            start_time_pred = time.time()
            test_pred = pipe.predict(cb_test_X)
            test_pred_time = time.time() - start_time_pred
            
            start_time_pred_train = time.time()
            train_pred = pipe.predict(cb_train_X)
            train_pred_time = time.time() - start_time_pred_train
            
            test_rmse = np.sqrt(mean_squared_error(test_y, test_pred))
            test_r2 = r2_score(test_y, test_pred)
            test_mae = mean_absolute_error(test_y, test_pred)
            test_mape = mean_absolute_percentage_error(test_y, test_pred)
            
            train_rmse = np.sqrt(mean_squared_error(train_y, train_pred))
            train_r2 = r2_score(train_y, train_pred)
            train_mae = mean_absolute_error(train_y, train_pred)
            train_mape = mean_absolute_percentage_error(train_y, train_pred)
            
            test_preds_df[model_name] = test_pred
            train_preds_df[model_name] = train_pred

        else:
            start_time_fit = time.time()
            pipe.fit(train_X, train_y)
            fit_time = time.time() - start_time_fit
            
            start_time_pred = time.time()
            test_pred = pipe.predict(test_X)
            test_pred_time = time.time() - start_time_pred
            
            start_time_pred_train = time.time()
            train_pred = pipe.predict(train_X)
            train_pred_time = time.time() - start_time_pred_train
            
            test_rmse = np.sqrt(mean_squared_error(test_y, test_pred))
            test_r2 = r2_score(test_y, test_pred)
            test_mae = mean_absolute_error(test_y, test_pred)
            test_mape = mean_absolute_percentage_error(test_y, test_pred)
            
            train_rmse = np.sqrt(mean_squared_error(train_y, train_pred))
            train_r2 = r2_score(train_y, train_pred)
            train_mae = mean_absolute_error(train_y, train_pred)
            train_mape = mean_absolute_percentage_error(train_y, train_pred)
            
            test_preds_df[model_name] = test_pred
            train_preds_df[model_name] = train_pred

        # Uložení všech metrik do finálního seznamu
        results_scores_test.append({
            "Dataset": dataset_size_name, "Model": model_name, 
            "RMSE": test_rmse, "R2": test_r2, "MAE": test_mae, "MAPE": test_mape,
            "Fit Time (s)": fit_time, "Pred Time (s)": test_pred_time
        })
        results_scores_train.append({
            "Dataset": dataset_size_name, "Model": model_name, 
            "RMSE": train_rmse, "R2": train_r2, "MAE": train_mae, "MAPE": train_mape,
            "Fit Time (s)": fit_time, "Pred Time (s)": train_pred_time
        })

    df_scores_test = pd.DataFrame(results_scores_test)
    df_scores_train = pd.DataFrame(results_scores_train)

    test_preds_path = os.path.join(out_dir_preds, f"test_preds_{dataset_size_name}_{tuning_status}.parquet")
    train_preds_path = os.path.join(out_dir_preds, f"train_preds_{dataset_size_name}_{tuning_status}.parquet")
    test_scores_path = os.path.join(out_dir_scores, f"test_scores_{dataset_size_name}_{tuning_status}.csv")
    train_scores_path = os.path.join(out_dir_scores, f"train_scores_{dataset_size_name}_{tuning_status}.csv")

    test_preds_df.to_parquet(test_preds_path)
    train_preds_df.to_parquet(train_preds_path)
    df_scores_test.to_csv(test_scores_path, index=False)
    df_scores_train.to_csv(train_scores_path, index=False)

    print(f"Finished evaluating {dataset_size_name} dataset ({tuning_status}). Results saved to {out_dir_scores}/ and {out_dir_preds}/.")

    return df_scores_test, df_scores_train, test_preds_df, train_preds_df

In [17]:
compare_models_reg(models_no_tuning_reg, train_1k_X, train_1k_y_reg, test_full_X, test_full_y_reg, "1k", "not_tuned")

Evaluating XGBoost on 1k dataset (not_tuned)...
Evaluating LightGBM on 1k dataset (not_tuned)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000707 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6787
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 83
[LightGBM] [Info] Start training from score -0.064360
Evaluating CatBoost on 1k dataset (not_tuned)...
Evaluating NGBoost on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating PGBM on 1k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktick

Evaluating Dummy - Most Frequent on 1k dataset (not_tuned)...
Finished evaluating 1k dataset (not_tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


(  Dataset                  Model      RMSE        R2       MAE          MAPE  \
 0      1k                XGBoost  0.388365 -0.136943  0.275853  1.729661e+11   
 1      1k               LightGBM  0.374999 -0.060031  0.271866  1.519567e+11   
 2      1k               CatBoost  0.366697 -0.013615  0.266525  1.155219e+11   
 3      1k                NGBoost  0.365601 -0.007564  0.267916  1.025272e+11   
 4      1k                    GBM  0.370054 -0.032260  0.268450  1.341662e+11   
 5      1k                HistGBM  0.373327 -0.050601  0.272272  1.413219e+11   
 6      1k                   PGBM  0.372736 -0.047278  0.272934  1.609742e+11   
 7      1k  Dummy - Most Frequent  0.364261 -0.000195  0.280289  1.014339e+11   
 
    Fit Time (s)  Pred Time (s)  
 0      0.316629       0.160174  
 1      0.381519       0.529290  
 2      2.320436       0.463112  
 3      8.932232      23.753744  
 4      0.774862       1.338839  
 5      0.767223       0.769007  
 6      1.827946       0.738490

In [18]:
compare_models_reg(models_no_tuning_reg, train_10k_X, train_10k_y_reg, test_full_X, test_full_y_reg, "10k", "not_tuned")

Evaluating XGBoost on 10k dataset (not_tuned)...
Evaluating LightGBM on 10k dataset (not_tuned)...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001435 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 8167
[LightGBM] [Info] Number of data points in the train set: 10000, number of used features: 87
[LightGBM] [Info] Start training from score -0.031741
Evaluating CatBoost on 10k dataset (not_tuned)...
Evaluating NGBoost on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/pipeline.py:62: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will 

Evaluating GBM on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating HistGBM on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Evaluating PGBM on 10k dataset (not_tuned)...


/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktick

Evaluating Dummy - Most Frequent on 10k dataset (not_tuned)...
Finished evaluating 10k dataset (not_tuned). Results saved to test_results/Regression/scores/ and test_results/Regression/predictions/.


(  Dataset                  Model      RMSE        R2       MAE          MAPE  \
 0     10k                XGBoost  0.377267 -0.072896  0.261333  1.692373e+11   
 1     10k               LightGBM  0.362031  0.012013  0.248470  8.394833e+10   
 2     10k               CatBoost  0.360329  0.021280  0.249037  9.235742e+10   
 3     10k                NGBoost  0.359907  0.023571  0.249641  5.905795e+10   
 4     10k                    GBM  0.360356  0.021138  0.249655  6.474612e+10   
 5     10k                HistGBM  0.361786  0.013352  0.247775  8.047953e+10   
 6     10k                   PGBM  0.361951  0.012451  0.247473  8.460189e+10   
 7     10k  Dummy - Most Frequent  0.365265 -0.005714  0.261587  5.002489e+10   
 
    Fit Time (s)  Pred Time (s)  
 0      0.404224       0.130553  
 1      0.463957       0.268946  
 2      4.681586       0.442855  
 3     75.641931      22.274966  
 4      7.257874       1.410014  
 5      0.871673       0.718347  
 6      1.917139       0.578703

In [ ]:
compare_models_reg(models_no_tuning_reg, train_100k_X, train_100k_y_reg, test_full_X, test_full_y_reg, "100k", "not_tuned")

Evaluating XGBoost on 100k dataset (not_tuned)...
Evaluating LightGBM on 100k dataset (not_tuned)...
[LightGBM] [Info] Number of positive: 25346, number of negative: 74654
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.015694 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9137
[LightGBM] [Info] Number of data points in the train set: 100000, number of used features: 92
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.253460 -> initscore=-1.080243
[LightGBM] [Info] Start training from score -1.080243
Evaluating CatBoost on 100k dataset (not_tuned)...
Evaluating NGBoost on 100k dataset (not_tuned)...


In [ ]:
compare_models_reg(models_no_tuning_reg, train_full_X, train_full_y_reg, test_full_X, test_full_y_reg, "full", "not_tuned")

In [ ]:
compare_models_reg(models_tuned_1k_reg, train_1k_X, train_1k_y_reg, test_full_X, test_full_y_reg, "1k", "tuned")

In [ ]:
compare_models_reg(models_tuned_10k_reg, train_10k_X, train_10k_y_reg, test_full_X, test_full_y_reg, "10k", "tuned")

In [ ]:
compare_models_reg(models_tuned_100k_reg, train_100k_X, train_100k_y_reg, test_full_X, test_full_y_reg, "100k", "tuned")

In [ ]:
compare_models_reg(models_tuned_full_reg, train_full_X, train_full_y_reg, test_full_X, test_full_y_reg, "full", "tuned")